# Plot one CellChat ligand–receptor feature by pathology

Set the ligand, receptor, region, sender, and receiver in the parameter cell, then run the notebook from top to bottom. The plot follows the finalized EXP002 selected-feature style: donor points by pathology, the fitted pathology trend, model slope and confidence interval, pair-specific BH q value, and donor counts.

Scores are reconstructed from finalized nonzero donor probabilities. Missing feature rows are set to zero only for completed networks where both sender and receiver groups were eligible, following the EXP002 convention. No tests are refit. Set `require_significant = True` if you want to plot only features with pair-specific BH q < 0.05.

In [ ]:
# Editable feature and figure parameters
ligand = "SPP1"
receptor = "CD44"  # For example: ITGAV_ITGB5, ITGA9_ITGB1, or CD44
region = "EC"
sender = "Oli"
receiver = "Ast"

figure_width = 3.8 # inches
figure_height = 3.55  # inches
require_significant = False  # If True, require pair-specific BH q < 0.05
save_figure = True


In [ ]:
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Find the project root whether Jupyter starts in the repository or this notebook folder.
EXP = (Path(os.environ["REGA_SHARED_ROOT"]) / "part8_cellchat") if os.environ.get("REGA_SHARED_ROOT") else next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "finalized/tables/pathology_progression_tests.tsv").exists())
FINAL = EXP / "finalized"
PATHOLOGY = ["nonAD", "earlyAD", "lateAD"]
COLORS = {"nonAD": "#4477AA", "earlyAD": "#EE9944", "lateAD": "#CC6677"}
interaction_id = f"{ligand.strip()}_{receptor.strip()}"

tests = pd.read_csv(FINAL / "tables/pathology_progression_tests.tsv", sep="\t")
selected = tests.loc[
    tests["region"].astype(str).str.upper().eq(region.strip().upper())
    & tests["source"].astype(str).eq(sender.strip())
    & tests["target"].astype(str).eq(receiver.strip())
    & tests["interaction_id"].astype(str).eq(interaction_id)
].copy()
if require_significant:
    selected = selected.loc[selected["q_value_within_family"].lt(0.05)].copy()
if len(selected) != 1:
    raise ValueError(f"Expected one matching test row for {region}: {sender} -> {receiver}, {interaction_id}; found {len(selected)}. Check names or turn off require_significant.")
feature = selected.iloc[0]
print(f"Selected: {region}: {sender} -> {receiver}, {interaction_id}")
print(f"Pair-specific BH q = {feature.q_value_within_family:.4g}")


In [ ]:
# Reconstruct eligible donor scores, using zero only for an eligible donor
# whose completed network has no nonzero row for this interaction.
status = pd.read_csv(str(EXP / "tables/donor_network_status.tsv"), sep="\t")
coverage_path =EXP / f"tables/{region.strip().upper()}_donor_celltype_coverage.tsv"
coverage = pd.read_csv(coverage_path, sep="\t")
coverage = coverage.loc[
    coverage["region"].astype(str).str.upper().eq(region.strip().upper())
    & coverage["retained_for_network"].astype(str).str.lower().eq("true"),
    ["donor_id", "celltype"],
]
completed = status.loc[
    status["region"].astype(str).str.upper().eq(region.strip().upper())
    & status["donor_network_status"].eq("completed"),
    ["region", "donor_id", "pathology"],
].drop_duplicates()
if completed["donor_id"].duplicated().any():
    raise ValueError(f"Duplicate completed-network donor records for {region}")
eligible = completed.merge(
    coverage.loc[coverage["celltype"].eq(sender.strip()), ["donor_id"]].assign(source_eligible=True),
    on="donor_id", how="inner",
).merge(
    coverage.loc[coverage["celltype"].eq(receiver.strip()), ["donor_id"]].assign(target_eligible=True),
    on="donor_id", how="inner",
)
nonzero_path = EXP / f"intermediate/{region.strip().upper()}_donor_nonzero_probabilities.tsv.gz"
nonzero = pd.read_csv(nonzero_path, sep="\t")
observed = nonzero.loc[
    nonzero["region"].astype(str).str.upper().eq(region.strip().upper())
    & nonzero["source"].eq(sender.strip())
    & nonzero["target"].eq(receiver.strip())
    & nonzero["interaction_id"].eq(interaction_id),
    ["donor_id", "communication_probability"],
]
if observed["donor_id"].duplicated().any():
    raise ValueError(f"Duplicate donor scores for {interaction_id}")
scores = eligible.merge(observed, on="donor_id", how="left", validate="one_to_one")
scores = scores.rename(columns={"communication_probability": "score"})
scores["score_row_present"] = scores["score"].notna()
scores["score"] = scores["score"].fillna(0.0)
scores["region"] = region.strip().upper()
scores["source"] = sender.strip()
scores["target"] = receiver.strip()
scores["interaction_id"] = interaction_id

counts = scores.groupby("pathology")["donor_id"].nunique().reindex(PATHOLOGY, fill_value=0)
expected_counts = [feature.n_nonAD, feature.n_earlyAD, feature.n_lateAD]
if counts.tolist() != expected_counts:
    raise ValueError(f"Eligible donor counts {counts.tolist()} do not match finalized tests {expected_counts}")
means = scores.groupby("pathology")["score"].mean().reindex(PATHOLOGY)
expected_means = [feature.mean_nonAD, feature.mean_earlyAD, feature.mean_lateAD]
if not np.allclose(means.to_numpy(float), expected_means, rtol=1e-7, atol=1e-9):
    raise ValueError("Reconstructed donor-score means do not match finalized tests")
display(pd.DataFrame({"n_donors": counts, "mean_score": means}))
print(f"Eligible donors: {len(scores)}; observed nonzero rows: {int(scores.score_row_present.sum())}; eligible zeros: {int((~scores.score_row_present).sum())}")


In [ ]:
# Plot donor scores and the finalized fitted pathology trend.
fig, ax = plt.subplots(figsize=(figure_width, figure_height))
rng = np.random.default_rng(20261001)
for x, pathology in enumerate(PATHOLOGY):
    values = scores.loc[scores["pathology"].eq(pathology), "score"].to_numpy(float)
    ax.scatter(x + rng.normal(0, 0.055, len(values)), values,
               color=COLORS[pathology], alpha=0.8, s=35,
               edgecolor="white", linewidth=0.4, zorder=3)
    ax.text(x, 0.98, f"n={len(values)}", transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=9)
predicted = [feature.prediction_nonAD_mean_covars,
             feature.prediction_earlyAD_mean_covars,
             feature.prediction_lateAD_mean_covars]
if np.isfinite(np.asarray(predicted, dtype=float)).all():
    ax.plot(range(3), predicted, color="#777777", linestyle="--",
            linewidth=1.8, zorder=2, label="Fitted pathology trend")
beta = "β=NA" if pd.isna(feature.beta_pathology_stage) else (
    f"β={feature.beta_pathology_stage:.3g} [{feature.ci_low:.3g}, {feature.ci_high:.3g}]"
)
qvalue = "BH q=NA" if pd.isna(feature.q_value_within_family) else f"BH q={feature.q_value_within_family:.3g}"
ax.set_title(f"{region.upper()}: {sender} → {receiver}\n{interaction_id}\n{beta}; {qvalue}")
ax.set_xticks(range(3), PATHOLOGY, rotation=15, ha="right")
ax.set_ylabel("CellChat probability")
ax.grid(axis="y", color="#dddddd", linewidth=0.55)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()


In [ ]:
from pathlib import Path
import re

if save_figure:
    output_dir = FINAL / "figures/single_significant_features"
    output_dir.mkdir(parents=True, exist_ok=True)

    safe = lambda value: re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))
    stem = output_dir / (
        f"{safe(region)}_{safe(sender)}_to_{safe(receiver)}_{safe(interaction_id)}"
    )

    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(stem.with_suffix(".png"), dpi=300, bbox_inches="tight")
    print(f"Saved {stem}.pdf and {stem}.png")

plt.close(fig)

## Batch plot all significant non-Vasc features

Run this cell to create one PDF per feature with pair-specific BH q < 0.05, excluding any sender or receiver pair involving Vasc. It uses the same donor-score reconstruction, count/mean consistency checks, model annotations, and plot style as the single-feature cells above. PDFs are written to `finalized/figures/single_significant_features/`.

In [ ]:
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EXP = (Path(os.environ["REGA_SHARED_ROOT"]) / "part8_cellchat") if os.environ.get("REGA_SHARED_ROOT") else next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "finalized/tables/pathology_progression_tests.tsv").exists())
FINAL = EXP / "finalized"
OUTPUT_DIR = FINAL / "figures" / "single_significant_features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PATHOLOGY = ["nonAD", "earlyAD", "lateAD"]
COLORS = {"nonAD": "#4477AA", "earlyAD": "#EE9944", "lateAD": "#CC6677"}

# Follow the finalized notebook convention: pair-specific BH q < 0.05.
tests = pd.read_csv(FINAL / "tables/pathology_progression_tests.tsv", sep="\t")
significant = tests.loc[
    tests["q_value_within_family"].lt(0.05)
    & tests["source"].astype(str).ne("Vasc")
    & tests["target"].astype(str).ne("Vasc")
].copy()
significant = significant.sort_values(["region", "source", "target", "interaction_id"]).reset_index(drop=True)

status_all = pd.read_csv(EXP / "tables/donor_network_status.tsv", sep="\t")

def safe(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))

saved = []
for region, region_tests in significant.groupby("region", sort=True):
    region = str(region).upper()
    coverage = pd.read_csv(EXP / f"tables/{region}_donor_celltype_coverage.tsv", sep="\t")
    coverage = coverage.loc[
        coverage["region"].astype(str).str.upper().eq(region)
        & coverage["retained_for_network"].astype(str).str.lower().eq("true"),
        ["donor_id", "celltype"],
    ]
    status = status_all.loc[
        status_all["region"].astype(str).str.upper().eq(region)
        & status_all["donor_network_status"].eq("completed"),
        ["region", "donor_id", "pathology"],
    ].drop_duplicates()
    if status["donor_id"].duplicated().any():
        raise ValueError(f"Duplicate completed-network donor records for {region}")
    nonzero = pd.read_csv(EXP / f"intermediate/{region}_donor_nonzero_probabilities.tsv.gz", sep="\t")

    for _, feature in region_tests.iterrows():
        sender = str(feature["source"])
        receiver = str(feature["target"])
        interaction_id = str(feature["interaction_id"])
        eligible = status.merge(
            coverage.loc[coverage["celltype"].eq(sender), ["donor_id"]].assign(source_eligible=True),
            on="donor_id", how="inner",
        ).merge(
            coverage.loc[coverage["celltype"].eq(receiver), ["donor_id"]].assign(target_eligible=True),
            on="donor_id", how="inner",
        )
        observed = nonzero.loc[
            nonzero["region"].astype(str).str.upper().eq(region)
            & nonzero["source"].eq(sender)
            & nonzero["target"].eq(receiver)
            & nonzero["interaction_id"].eq(interaction_id),
            ["donor_id", "communication_probability"],
        ]
        if observed["donor_id"].duplicated().any():
            raise ValueError(f"Duplicate donor scores for {region} {sender}->{receiver} {interaction_id}")
        scores = eligible.merge(observed, on="donor_id", how="left", validate="one_to_one")
        scores = scores.rename(columns={"communication_probability": "score"})
        scores["score"] = scores["score"].fillna(0.0)
        counts = scores.groupby("pathology")["donor_id"].nunique().reindex(PATHOLOGY, fill_value=0)
        expected_counts = [int(feature.n_nonAD), int(feature.n_earlyAD), int(feature.n_lateAD)]
        if counts.tolist() != expected_counts:
            raise ValueError(f"{region} {sender}->{receiver} {interaction_id}: donor counts {counts.tolist()} != {expected_counts}")
        means = scores.groupby("pathology")["score"].mean().reindex(PATHOLOGY)
        expected_means = [feature.mean_nonAD, feature.mean_earlyAD, feature.mean_lateAD]
        if not np.allclose(means.to_numpy(float), expected_means, rtol=1e-7, atol=1e-9):
            raise ValueError(f"{region} {sender}->{receiver} {interaction_id}: reconstructed means differ from finalized tests")

        fig, ax = plt.subplots(figsize=(3.8, 3.55))
        rng = np.random.default_rng(20261001)
        for x, pathology in enumerate(PATHOLOGY):
            values = scores.loc[scores["pathology"].eq(pathology), "score"].to_numpy(float)
            ax.scatter(x + rng.normal(0, 0.055, len(values)), values,
                       color=COLORS[pathology], alpha=0.8, s=35,
                       edgecolor="white", linewidth=0.4, zorder=3)
            ax.text(x, 0.98, f"n={len(values)}", transform=ax.get_xaxis_transform(),
                    ha="center", va="top", fontsize=9)
        predicted = [feature.prediction_nonAD_mean_covars,
                     feature.prediction_earlyAD_mean_covars,
                     feature.prediction_lateAD_mean_covars]
        if np.isfinite(np.asarray(predicted, dtype=float)).all():
            ax.plot(range(3), predicted, color="#777777", linestyle="--",
                    linewidth=1.8, zorder=2, label="Fitted pathology trend")
        beta = "β=NA" if pd.isna(feature.beta_pathology_stage) else (
            f"β={feature.beta_pathology_stage:.3g} [{feature.ci_low:.3g}, {feature.ci_high:.3g}]"
        )
        qvalue = "BH q=NA" if pd.isna(feature.q_value_within_family) else f"BH q={feature.q_value_within_family:.3g}"
        ax.set_title(f"{region}: {sender} → {receiver}\n{interaction_id}\n{beta}; {qvalue}")
        ax.set_xticks(range(3), PATHOLOGY, rotation=15, ha="right")
        ax.set_ylabel("CellChat probability")
        ax.grid(axis="y", color="#dddddd", linewidth=0.55)
        ax.spines[["top", "right"]].set_visible(False)
        fig.tight_layout()
        output_path = OUTPUT_DIR / f"{safe(region)}_{safe(sender)}_to_{safe(receiver)}_{safe(interaction_id)}.pdf"
        fig.savefig(output_path, bbox_inches="tight")
        plt.close(fig)
        saved.append(output_path)
    print(f"{region}: saved {len(region_tests)} PDFs")

if len(saved) != len(significant):
    raise RuntimeError(f"Saved {len(saved)} PDFs for {len(significant)} significant tests")
print(f"Saved {len(saved)} individual PDFs to {OUTPUT_DIR}")